In [ ]:
import matplotlib.pyplot as plt
import os
import numpy as np
import json

In [ ]:
fontsize_legend = 16
fontsize_label = 22
fontsize_ticks = 18

In [ ]:
method2name = {
    'Random': 'Random',
    'RoundRobin': 'RoundRobin',
}
method2color = {
    'Random': [.8, .8, 0],
    'RoundRobin': [1, .6, 0],
    'STOP-V=1000': [1, 0, .3],
}

In [ ]:
def get_path(model, method, n_m, max_d, id, C_avg=None, R_avg=None):
    path = f'results_retry/{model}/{method}/n_m={n_m}-max_d={max_d}'
    if 'STOP' in method or 'Li' in method:
        path += f'/C_avg={C_avg}-R_avg={R_avg}'
    path += f'/id={id}'
    return path

In [ ]:
def get_data(path):
    data = dict()
    for t in range(300):
        if os.path.isfile(f'{path}/t={t}.json'):
            with open(f'{path}/t={t}.json', 'r') as f:
                d = json.load(f)
            keys = list(d.keys())
            for key in keys:
                if key not in data:
                    data[key] = list()
                data[key].append(d[key])
        else:
            print(f'File {path}/t={t}.json does not exist.')
            break
    for key in keys:
        try:
            data[key] = np.asarray(data[key]).astype('float32')
        except:
            pass
    with open(f'{path}/max_diff.txt', 'r') as f:
        overall_max_diff = float(f.readline().strip())
    response_time_k = np.mean(data['response_time_k'])
    return {'n_demands': data['arrivals'], 'dispatcher_queue': data['not_executed'], 'overall_max_diff': overall_max_diff, 'response_time_k': response_time_k}

In [ ]:
def open_data(model, methods, machines, seeds, max_d, C_avg, R_avg):
    method2n_m2data = dict()
    for method in methods:
        method2n_m2data[method] = dict()
        for n_m in machines:
            method2n_m2data[method][n_m] = dict()
            for s in seeds:
                d = get_data(get_path(model, method, n_m, max_d, s, C_avg, R_avg))
                for k in d:
                    if k not in method2n_m2data[method][n_m]:
                        method2n_m2data[method][n_m][k] = list()
                    method2n_m2data[method][n_m][k].append(d[k])
    return method2n_m2data

In [ ]:
machines = range(3, 8)
data = open_data('Yolo', ['STOP-V=1000'], machines, range(30), 3000, .003, 7)
for n_m in machines:
    data['STOP-V=1000'][n_m]['n_demands'] = np.asarray(data['STOP-V=1000'][n_m]['n_demands'])
    data['STOP-V=1000'][n_m]['dispatcher_queue'] = np.asarray(data['STOP-V=1000'][n_m]['dispatcher_queue'])

In [ ]:
for n_m in machines:
    print(f'maximum number of timesteps that a demand waited in the dispatcher queue (n_m = {n_m}):', max(data['STOP-V=1000'][n_m]['overall_max_diff']))

In [ ]:
for n_m in machines:
    plt.figure(figsize=(8, 1))
    mean = data['STOP-V=1000'][n_m]['dispatcher_queue'].mean(0)
    std = data['STOP-V=1000'][n_m]['dispatcher_queue'].std(0)
    plt.plot(mean, linewidth=2, label='Dispatcher Queue')
    plt.fill_between(range(300), mean - std, mean + std, alpha=0.2)
    plt.ylim([0, None])
    plt.xlim((0, 300))
    plt.xlabel('Time', fontsize=fontsize_label)
    plt.ylabel(r'$D(t)$', fontsize=fontsize_label)
    plt.xticks(fontsize=fontsize_ticks)
    plt.yticks(range(0, 1001, 500), fontsize=fontsize_ticks)
    # plt.legend(fontsize=fontsize_legend)
    plt.savefig(f'dispatcher_queue_n_m={n_m}.pdf', bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
# Little's law
for n_m in machines:
    N = data['STOP-V=1000'][n_m]['n_demands'].mean(1)
    arrival_rate = (data['STOP-V=1000'][n_m]['n_demands'] - data['STOP-V=1000'][n_m]['dispatcher_queue']).mean(1)
    print(f'Little law (n_m = {n_m}): {(N / arrival_rate - 1).mean().item():.2f}s')

In [ ]:
print('Response time:', np.mean(data['STOP-V=1000'][6]['response_time_k']).item())